In [1]:
import anthropic
import os
import pandas as pd
import duckdb
from dotenv import load_dotenv

In [2]:
conn = duckdb.connect('/app/data/analytics.duckdb')

In [3]:
load_dotenv()
client = anthropic.Anthropic(api_key=os.getenv('ANTHROPIC_API_KEY'))

In [4]:
with open('/app/data/batch_id.txt', 'r') as f:
    batch_id = f.read().strip()

In [5]:
batch = client.messages.batches.retrieve(batch_id)
print(f"Status: {batch.processing_status}")
print(f"Request counts: {batch.request_counts}")

Status: ended
Request counts: MessageBatchRequestCounts(canceled=0, errored=18, expired=0, processing=0, succeeded=48825)


In [6]:
if batch.processing_status == "ended":
    results = []
    for entry in client.messages.batches.results(batch_id):
        if entry.result.type == "succeeded":
            # Parse custom_id to get type and review_id
            custom_id = entry.custom_id
            if custom_id.startswith('msg_'):
                review_id = custom_id[4:]
                results.append({
                    "review_id": review_id, 
                    "type": "message",
                    "translated_text": entry.result.message.content[0].text
                })
            elif custom_id.startswith('title_'):
                review_id = custom_id[6:]
                results.append({
                    "review_id": review_id,
                    "type": "title",
                    "translated_text": entry.result.message.content[0].text
                })

    results_df = pd.DataFrame(results)
    
    # Pivot so messages and titles are separate columns
    messages_df = results_df[results_df['type'] == 'message'][['review_id', 'translated_text']].rename(columns={'translated_text': 'translated_message'})
    titles_df = results_df[results_df['type'] == 'title'][['review_id', 'translated_text']].rename(columns={'translated_text': 'translated_title'})
    
    # Merge
    translations_df = messages_df.merge(titles_df, on='review_id', how='outer')
    
    print(f"Total reviews translated: {len(translations_df):,}")
    print(f"With messages: {translations_df['translated_message'].notna().sum():,}")
    print(f"With titles: {translations_df['translated_title'].notna().sum():,}")
    
    # Save to DuckDB
    conn = duckdb.connect('/app/data/analytics.duckdb')
    conn.execute("CREATE SCHEMA IF NOT EXISTS llm_outputs")
    conn.execute("DROP TABLE IF EXISTS llm_outputs.translated_reviews")
    conn.execute("CREATE TABLE llm_outputs.translated_reviews AS SELECT * FROM translations_df")
    
    print(f"\nSaved to llm_outputs.translated_reviews in DuckDB")
    
    # Also save to CSV as backup
    translations_df.to_csv('/app/data/translated_reviews.csv', index=False)
    print("Saved to /app/data/translated_reviews.csv as backup")

else:
    print(f"Batch not ready yet. Status: {batch.processing_status}")
    print(f"Request counts: {batch.request_counts}")

Total reviews translated: 39,946
With messages: 37,718
With titles: 11,107

Saved to llm_outputs.translated_reviews in DuckDB
Saved to /app/data/translated_reviews.csv as backup
